In [1]:
import os
from glob import glob
import torch
import numpy as np
import random

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

X_train = np.stack([np.loadtxt(file, dtype=np.float32) for file in glob(os.path.join("train/Inertial Signals", "*.txt"))], axis=-1)
y_train = np.loadtxt("train/y_train.txt", dtype=np.int64) - 1
X_test = np.stack([np.loadtxt(file, dtype=np.float32) for file in glob(os.path.join("test/Inertial Signals", "*.txt"))], axis=-1)
y_test = np.loadtxt("test/y_test.txt", dtype=np.int64) - 1 
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((7352, 128, 9), (7352,), (2947, 128, 9), (2947,))

In [2]:
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from tqdm.auto import tqdm
from mytorch import CNNLSTM
from ResCBAR import ResCBAR
from Best import CNN

batch_size=64
epochs=100
device='cuda'

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test)), batch_size=batch_size, shuffle=False)
model = CNNLSTM(classes=6, dim=16, num_cnn=3, num_lstm=1).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters(), lr=1e-3)

model.train()
for epoch in tqdm(range(epochs)):
    train_losses, train_correct, train_total = [], 0, 0
    for train, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(train.to(device, non_blocking=True)).to('cpu')
        train_loss = criterion(outputs, labels)
        train_loss.backward()
        optimizer.step()
        train_losses.append(train_loss.item())

        # Accuracy
        predicted = torch.argmax(outputs, dim=1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total
    if ((epoch+1) * 10 % epochs == 0):
        print(f"Train: Epoch [{epoch+1}/{epochs}], Loss: {torch.tensor(train_losses).mean():.4f}, Accuracy: {train_acc:.4f}")

C:\Users\Ur451\Desktop\SSM\.venv\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


  0%|          | 0/100 [00:00<?, ?it/s]

C:\Users\Ur451\Desktop\SSM\.venv\Lib\site-packages\torch\autograd\graph.py:829: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:179.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Train: Epoch [10/100], Loss: 0.0962, Accuracy: 0.9576
Train: Epoch [20/100], Loss: 0.0725, Accuracy: 0.9680
Train: Epoch [30/100], Loss: 0.0558, Accuracy: 0.9752
Train: Epoch [40/100], Loss: 0.0530, Accuracy: 0.9755
Train: Epoch [50/100], Loss: 0.0392, Accuracy: 0.9849
Train: Epoch [60/100], Loss: 0.0357, Accuracy: 0.9856
Train: Epoch [70/100], Loss: 0.0223, Accuracy: 0.9906
Train: Epoch [80/100], Loss: 0.0229, Accuracy: 0.9922
Train: Epoch [90/100], Loss: 0.0154, Accuracy: 0.9935
Train: Epoch [100/100], Loss: 0.0174, Accuracy: 0.9927


In [3]:
from sklearn.metrics import accuracy_score

model.eval()
test_pred, test_losses = [], []
with torch.no_grad():
    for test, labels in test_loader:
        outputs = model(test.to(device, non_blocking=True)).to('cpu')
        test_pred.append(outputs)
        test_loss = criterion(outputs, labels)
        test_losses.append(test_loss.item())
print(f"Loss: {torch.tensor(test_losses).mean():.4f}, Accuracy: {accuracy_score(y_test, torch.cat(test_pred).argmax(dim=1)):.4f}")

Loss: 0.2786, Accuracy: 0.9562
